# 01 - Setup e fondamenti

Primo notebook del percorso didattico Nimbus: verifica dell'ambiente e nozioni minime per capire di cosa parlano i prossimi quattro.

## Cosa NON e' questo percorso

- **Non e' la pipeline Nimbus** ne' un suo prototipo.
- **Non e' autorevole sui contratti dati.** La fonte di verita' resta `docs/`.
- **Non produce previsioni pubblicabili.** Nessun output di `learning/` puo' essere presentato come forecast Nimbus.

## Cos'e' un modello NWP e perche' ha una griglia

Il GFS risolve equazioni fisiche su celle di circa 0,25° (~25 km alle nostre latitudini). Dentro una cella il modello ha un solo valore: la sua "montagna" e' una media, non la montagna vera.

## Run time, valid time, lead

Il modello parte da uno stato iniziale a `run_time` e proietta in avanti; `lead_hours` e' quanto avanti; `valid_time = run_time + lead_hours`. Il GFS gira alle 00/06/12/18 UTC.

## Perche' UTC

L'ora locale ha fusi e ora legale: due timestamp identici possono essere istanti diversi. Tutta la meteorologia lavora in UTC. In questo percorso ogni timestamp e' tz-aware.

## Cos'e' un GRIB e perche' non e' un CSV

Formato binario compresso per campi grigliati, con metadati per messaggio. Va aperto con una libreria dedicata; e' il motivo per cui serve `eccodes`.

## Verifica dell'ambiente

La cella seguente importa `common.py`, crea le directory dei dati e controlla che le librerie necessarie ai prossimi notebook siano installate e importabili, registrando tutto in un report.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # per importare common.py
import common

common.ensure_dirs()

report = {
    "python": sys.version.split()[0],
    "pacchetti": {},
    "eccodes_api": None,
    "cartopy_ok": False,
    "generato_il": datetime.now(timezone.utc).isoformat(),
}

problemi = []

for pacchetto in ["pandas", "pyarrow", "numpy", "xarray", "cfgrib",
                  "eccodes", "matplotlib", "Cartopy", "requests"]:
    try:
        report["pacchetti"][pacchetto] = metadata.version(pacchetto)
    except metadata.PackageNotFoundError:
        report["pacchetti"][pacchetto] = None
        problemi.append(f"{pacchetto} non installato")

print("Python:", report["python"])
if not report["python"].startswith("3.12"):
    problemi.append(
        f"Python {report['python']}: il percorso e' verificato su 3.12. "
        "Ricreare il venv con /opt/homebrew/bin/python3.12 -m venv learning/.venv"
    )
for nome, versione in report["pacchetti"].items():
    print(f"  {nome:12s} {versione or 'MANCANTE'}")

## Le due dipendenze binarie

`eccodes` e `cartopy` non sono puro Python: sono wrapper di librerie C (ECMWF ecCodes, GEOS/PROJ). Questo significa che possono installarsi correttamente con `pip` e poi non importarsi, se manca la libreria nativa sottostante. Per questo meritano un controllo dedicato, separato dal semplice controllo di versione fatto sopra.

In [ ]:
# eccodes: la libreria ECMWF che legge il GRIB.
try:
    import eccodes
    report["eccodes_api"] = eccodes.codes_get_api_version()
    print("eccodes OK, versione libreria:", report["eccodes_api"])
    import cfgrib  # noqa: F401
    print("cfgrib OK: xarray potra' aprire i GRIB")
except Exception as errore:
    problemi.append(f"eccodes/cfgrib non utilizzabili: {type(errore).__name__}: {errore}")
    print("PROBLEMA con eccodes/cfgrib:", errore)
    print("Rimedio: learning/.venv/bin/pip install --force-reinstall eccodes cfgrib")
    print("Se persiste su macOS: brew install eccodes, poi reinstallare i pacchetti.")

# cartopy: confini e proiezioni per le mappe.
try:
    import cartopy.crs as ccrs  # noqa: F401
    report["cartopy_ok"] = True
    print("cartopy OK: le mappe avranno coste e confini")
except Exception as errore:
    print("cartopy non disponibile:", errore)
    print("Il percorso resta eseguibile: le mappe perderanno i contorni,")
    print("ma stazioni e griglia restano leggibili.")

## Cosa hai ottenuto

Il report viene scritto su disco in `learning/data/00_env_report.json`: e' l'artefatto che i notebook successivi possono controllare per sapere se l'ambiente e' a posto, invece di scoprirlo a meta' esecuzione.

In [ ]:
with common.data_path("00_env_report.json").open("w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

if problemi:
    print("AMBIENTE INCOMPLETO:")
    for p in problemi:
        print("  -", p)
else:
    print("Ambiente verificato. Puoi passare al notebook 02.")
print("\nReport scritto in:", common.data_path("00_env_report.json"))

## Il limite di quello che hai fatto

Hai verificato che le librerie si importano, non che i dati siano scaricabili: le fonti remote possono essere irraggiungibili o aver cambiato formato, e lo scoprirai nel notebook 02. Il report registra un ambiente in un istante: se reinstalli qualcosa, va rieseguito.